In [ ]:
from textwrap import dedent
import kagglehub
import pandas as pd

from mistralai import Mistral
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from tqdm import tqdm

tqdm.pandas()

# Load dataset

In [ ]:
def load_tweets_data() -> pd.DataFrame:
  """
  Load the Tweets dataset from either the data folder or Kaggle.

  This function first attempts to load the dataset from a predefined
  path. If that fails, it falls back to downloading the dataset from
  Kaggle using the `kagglehub` package.

  Returns:
      pd.DataFrame: A DataFrame containing the tweet data from `Tweets.csv`.
  """
  file_name = "Tweets.csv"
  try:
    file_path = f"/data/{file_name}"
    return pd.read_csv(file_path)
  except:
    kaggle_dataset_name = "crowdflower/twitter-airline-sentiment"
    dataset_path = kagglehub.dataset_download(kaggle_dataset_name)
    return pd.read_csv(f"{dataset_path}/{file_name}")

In [ ]:
data = load_tweets_data()
print("Number of messages: ", len(data))
data.head()

In [ ]:
data = data.sample(frac=0.1, random_state=31).reset_index(drop=True)
print("Number of messages:", len(data))

In [ ]:
pd.set_option("display.max_colwidth", None)
data = data[["text", "airline_sentiment"]]
print(len(data))
data.head(10)

# APPROACH 1: EMBEDDINGS

## Transform texts into vectors

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(data["text"].tolist(), show_progress_bar=True, normalize_embeddings=True,)
embedding_df = pd.DataFrame(embeddings, columns=[f"embedding_{i}" for i in range(embeddings.shape[1])])
data = pd.concat([data, embedding_df], axis=1)
data.head()

## Train/test split

In [ ]:
data_train, data_test = train_test_split(data, test_size=0.2,
                                         stratify=data["airline_sentiment"],
                                         random_state=31)

X_train = data_train.drop(["text", "airline_sentiment"], axis=1)
y_train = data_train["airline_sentiment"]

X_test = data_test.drop(["text", "airline_sentiment"], axis=1)
y_test = data_test["airline_sentiment"]

print("Size of train:", len(y_train))
print("Size of test:", len(y_test))

## Train model

In [ ]:
classifier = LogisticRegression(max_iter=1000)
classifier.fit(X_train, y_train)

## Evaluate model

In [ ]:
y_pred = classifier.predict(X_test)
data_test = data_test[["text", "airline_sentiment"]].reset_index(drop=True)
data_test["prediction_embeddings"] = y_pred
data_test.head(10)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# APPROACH 2: LARGE LANGUAGE MODEL (LLM)

In [ ]:
model = "mistral-small-2503"
api_key = ""

client = Mistral(api_key=api_key)

def call_model(query: str) -> str:
  """
  Send a query to the chat model and return its response.

  This function uses the global `client` object to call the chat
  completion API with the specified model. The query is sent as a
  user message, and the first response message is returned.

  Args:
      query (str): The user input or prompt to send to the model.

  Returns:
      str: The content of the model's first response message.

  Raises:
      Exception: If the client call fails or no response is returned.
  """
  chat_response = client.chat.complete(
  model = model,
  messages = [{"role": "user", "content": query}])
  return chat_response.choices[0].message.content

In [ ]:
call_model("is the following message positive, negative or neutral?: I will use this airline again because the coffee is perfect. thank you!")

## Generate prompt for text classification

In [ ]:
prompt = "I will provide a tweet that someone wrote to an airline. Analyse the sentiment of the tweet and return only one word: positive, negative or neutral. Provided tweet: "

In [ ]:
call_model(prompt + "I will always use this airline again because it is the best one")

In [ ]:
data_test.head()

In [ ]:
data_test["prediction_llm"] = data_test["text"].progress_apply(lambda x: call_model(prompt + x))
data_test.head()

In [ ]:
y_pred = data_test["prediction_llm"]
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## Improve prompt

In [ ]:
prompt_improved = dedent("""
    You are an expert sentiment analyst specializing in customer service interactions with airlines.
    Your task is to classify the sentiment of a tweet directed at an airline.

    Follow these steps (think through them privately before answering, but only output the final classification word):
    1. Read the tweet carefully.
    2. Consider whether the tone is positive, negative, or neutral.
    3. Output exactly one word: positive, negative, or neutral. Do not add anything else.

    Examples:
    Tweet: "Thank you @Delta for the smooth flight, best crew ever!"
    Answer: positive

    Tweet: "My flight with @United was delayed for 6 hours and no updates were given."
    Answer: negative

    Tweet: "@AmericanAir what time does flight 123 depart from JFK?"
    Answer: neutral

    Now classify the following tweet:
""")

In [ ]:
data_test["prediction_llm2"] = data_test["text"].progress_apply(lambda x: call_model(prompt_improved + x))
data_test.head()

In [ ]:
y_pred = data_test["prediction_llm2"]
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))